
**Objetivo del notebook**

Determinar si el grupo no fatales presenta suficiente heterogeneidad en sus características para justificar la aplicación de K-Modes.

**Pregunta de investigación:** ¿Existe evidencia suficiente para pensar que el grupo de no fatales puede dividirse en perfiles diferenciados utilizando las siete variables categóricas?

Para dar respuesta a la pregunta de investigación se plantea:

1. El numero de perfiles distintos.

2. El promedio de personas por perfil.

3. Se calcula la Entropía de Shannon para establecer la diversidad de las variables categóricas. Entropía baja significa casi todos los casos pertenecen a una sola categoría, Entropía alta las categorías están más equilibradas. Se normaliza la entropía para facilitar la interpretación de la medida, es decir, valores cercanos a 0 menor diversidad de casos en categorías de variables y 1 mayor diversidad.

4. Se calcula la Matriz de V de Cramér para establecer la asociación entre variables categóricas. va de 0 a 1, entre más cercana a 1 es más fuerte la asociación entre variables.

-Entrada:

Base de datos: nofatales_muerte_var_significativas, cuyo origen es el notebook 01_7.

Interpretación: al final del notebook.





In [0]:
#Importar librerías
from pyspark.sql import functions as F
from pyspark.sql.functions import col
import pandas as pd
import matplotlib.pyplot as plt
import math
from scipy.stats import chi2_contingency
import numpy as np


In [0]:
#Leer la tabla Delta

tabla = "ml_proyecto_7405607705157039.default.nofatales_muerte_var_significativas"

df = spark.table(tabla)

In [0]:
#Verificar estructura
df.printSchema()

In [0]:
display(df)

In [0]:
df.groupBy("grupo").count().show()

In [0]:
#Separar ambos grupos

df_nofatales = df.filter(col("grupo")=="nofatales")
df_muerte = df.filter(col("grupo")=="muerte")

In [0]:
#Variables del estudio

variables = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "presunto_agresor_cod"

]

In [0]:
#Número de categorías distintas

for v in variables:

    print(v)

    print(df_nofatales.select(v).distinct().count())

In [0]:
#verificar registros duplicados
#existen muchos individuos con exactamente el mismo perfil categórico

duplicados = (

    df_nofatales

    .groupBy(*variables)

    .count()

    .filter(col("count")>1)

    .count()

)

print("Combinaciones repetidas:",duplicados)

In [0]:
#tablas de frecuencias por variable

def tabla_frecuencias(df, variable, total_registros):
    return (
        df.groupBy(variable)
          .count()
          .withColumn(
              "Porcentaje",
              F.round(col("count") * 100 / total_registros, 2)
          )
          .orderBy(F.desc("count"))
    )

In [0]:
for variable in variables:

    print(f"\n{'='*80}")
    print(variable)
    print(f"{'='*80}")

    tabla = tabla_frecuencias(df_nofatales, variable, total_nofatales)

    display(tabla)

In [0]:
#Gráficas de frecuencia

for variable in variables:

    tabla_frecuencias = (
        df_nofatales
        .groupBy(variable)
        .count()
        .orderBy(F.desc("count"))
    )

    pdf = tabla_frecuencias.toPandas()

    plt.figure(figsize=(10,5))

    barras = plt.bar(pdf[variable], pdf["count"])

    plt.title(f"Distribución de {variable}")
    plt.xlabel(variable)
    plt.ylabel("Frecuencia")

    plt.xticks(rotation=90)
    plt.grid(axis="y", alpha=0.3)

    # Etiquetas sobre las barras
    for barra in barras:
        altura = barra.get_height()
        plt.text(
            barra.get_x() + barra.get_width()/2,
            altura,
            f"{int(altura):,}",
            ha="center",
            va="bottom",
            fontsize=8
        )

    plt.tight_layout()
    plt.show()

In [0]:
#Diversidad de las variables categóricas

# se utiliza la Entropía de Shannon para establecer la diversidad de las variables categóricas.
#Entropía baja → casi todos los casos pertenecen a una sola categoría.
#Entropía alta → las categorías están más equilibradas.

#Función para calcular la Entropía de Shannon

def calcular_entropia(df, variable, total_registros):

    # Frecuencia de cada categoría
    frecuencias = (
        df.groupBy(variable)
          .count()
          .collect()
    )

    # Número de categorías
    k = len(frecuencias)

    # Calcular entropía
    entropia = 0

    for fila in frecuencias:

        p = fila["count"] / total_registros

        entropia -= p * math.log2(p)

    # Entropía máxima
    entropia_maxima = math.log2(k) if k > 1 else 0

    # Entropía normalizada
    if entropia_maxima == 0:
        entropia_normalizada = 0
    else:
        entropia_normalizada = entropia / entropia_maxima

    return k, entropia, entropia_maxima, entropia_normalizada

In [0]:
#Entropía de todas las variables

resultados = []

for variable in variables:

    k, H, Hmax, Hnorm = calcular_entropia(
        df_nofatales,
        variable,
        total_nofatales
    )

    resultados.append({

        "Variable": variable,

        "Categorias": k,

        "Entropía": round(H,4),

        "Entropía máxima": round(Hmax,4),

        "Entropía normalizada": round(Hnorm,4)

    })

df_entropia = (
    pd.DataFrame(resultados)
      .sort_values(
          by="Entropía normalizada",
          ascending=False
      )
)

display(df_entropia)

In [0]:
#Asociación entre variables 

# Función Chi-cuadrado y V de Cramér

def calcular_cramers_v(df, variable1, variable2):

    # Tabla de contingencia
    tabla = (
        df.groupBy(variable1)
          .pivot(variable2)
          .count()
          .fillna(0)
          .toPandas()
    )

    # Matriz de frecuencias
    matriz = tabla.iloc[:,1:].values

    # Chi-cuadrado
    chi2, p, gl, esperado = chi2_contingency(matriz)

    # Número de observaciones
    n = matriz.sum()

    # Dimensiones de la tabla
    r, k = matriz.shape

    # V de Cramér
    v = np.sqrt(chi2 / (n * (min(r-1, k-1))))

    return chi2, p, gl, v

In [0]:
#Calcular todas las combinaciones

resultados = []

for i in range(len(variables)):

    for j in range(i+1, len(variables)):

        var1 = variables[i]
        var2 = variables[j]

        chi2, p, gl, v = calcular_cramers_v(
            df_nofatales,
            var1,
            var2
        )

        resultados.append({

            "Variable 1": var1,

            "Variable 2": var2,

            "Chi2": round(chi2,2),

            "gl": gl,

            "p-valor": round(p,5),

            "V de Cramér": round(v,4)

        })

df_cramer = (
    pd.DataFrame(resultados)
      .sort_values(
          by="V de Cramér",
          ascending=False
      )
)

display(df_cramer)

In [0]:
#Construir una matriz de asociación

# Matriz de V de Cramér
#V de Cramér, va de 0 a 1, entre más cercana a 1 es más fuerte la asociación entre variables

matriz = pd.DataFrame(
    np.eye(len(variables)),
    index=variables,
    columns=variables
)

for _, fila in df_cramer.iterrows():

    v = fila["V de Cramér"]

    matriz.loc[fila["Variable 1"], fila["Variable 2"]] = v
    matriz.loc[fila["Variable 2"], fila["Variable 1"]] = v

display(matriz)

In [0]:
#Construcción de perfiles

df_perfiles = (

    df_nofatales

    .groupBy(*variables)

    .count()

    .withColumnRenamed("count","Frecuencia"))

In [0]:
display(
    df_perfiles.orderBy(F.desc("Frecuencia"))
)

In [0]:
#Número de perfiles unicos

numero_perfiles = df_perfiles.count()

print(f"Número de perfiles distintos: {numero_perfiles:,}")

In [0]:
print(f"Registros totales: {total_nofatales:,}")

In [0]:
promedio = total_nofatales/numero_perfiles

print(f"Promedio de personas por perfil: {promedio:.2f}")

In [0]:
#Distribución del tamaño de los perfiles

distribucion = (

    df_perfiles

    .groupBy("Frecuencia")

    .count()

    .orderBy("Frecuencia")

)

display(distribucion)

In [0]:
#Histograma

pdf = distribucion.toPandas()

plt.figure(figsize=(10,5))

plt.bar(
    pdf["Frecuencia"],
    pdf["count"]
)

plt.xlabel("Número de personas por perfil")

plt.ylabel("Cantidad de perfiles")

plt.title("Distribución de perfiles categóricos")

plt.grid(axis="y",alpha=0.3)

plt.show()

In [0]:
#Perfiles más frecuentes

display(

    df_perfiles

    .orderBy(F.desc("Frecuencia"))

    .limit(20)

)

In [0]:
#Cobertura de los perfiles

#¿Qué porcentaje de personas pertenece a los perfiles más frecuentes?

pdf = (

    df_perfiles

    .orderBy(F.desc("Frecuencia"))

    .toPandas()
)

In [0]:
pdf["Acumulado"] = pdf["Frecuencia"].cumsum()

pdf["Porcentaje"] = (
    pdf["Acumulado"]/
    total_nofatales*100
)

In [0]:
plt.figure(figsize=(10,5))

plt.plot(pdf["Porcentaje"])

plt.grid()

plt.ylabel("% acumulado")

plt.xlabel("Número de perfiles")

plt.title("Cobertura acumulada de perfiles")

plt.show()

Interpretación:


Número de registros analizados:
643291

Número de variables categóricas:
7

Número de perfiles distintos:
4234

Promedio de personas por perfil:
151.93

Variable con mayor diversidad:
mecanismo_causal_cod, escolaridad_cod y presunto_agresor_cod, con entropía normalizada, 0.892, 0.8275 y 0.8029, respectivamente.

Variable con menor diversidad:
estado_civil_cod, con entropía normalizada: 0.6893.

Mayor asociación encontrada:
ciclo_vital_cod - Contexto del hecho
V de Cramér = 0,7356

Conclusión:

Los resultados evidencian una estructura heterogénea en el grupo de víctimas no fatales que justifican la aplicación de un algoritmo de agrupamiento para datos categóricos como K-Modes.